# TOP-10 ФП/СФП по окнам до default_date

## Цель
Построить TOP-10 ФП/СФП в разрезе сегментов (`МкБ`, `МСБ`, `ККБ`) для окон до даты дефолта:
- 2 года (`24m`)
- 1.5 года (`18m`)
- 1 год (`12m`)
- полгода (`6m`)

Дефолтная когорта: любой ИНН, присутствующий в `default_data.pkl`.

## Важная логика (простое объяснение)
Для каждого дефолтного ИНН формируется отдельный пул ФП/СФП в каждом окне.

Если в окне ФП/СФП нет, ставим `None`, но ИНН **не исключаем**.

Пример: в сегменте 5 дефолтных ИНН, у 2 из них в окне нет факторов (`None`).
Если фактор `13` встречается у 2 ИНН, то:
- `count_inn = 2`
- доля = `2 / 5 = 40%` (знаменатель — все 5 ИНН, включая `None`).

Так доли не завышаются и сравнение окон остается корректным.

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)

DATA_DIR = Path("/home/jovyan/documents/DMUKP_FP_DEF/data")

DATE_FROM = pd.Timestamp("2023-01-01")
DATE_TO = pd.Timestamp("2025-12-31")

SEGMENT_MAP = {
    "ДМСБ (микро)": "МкБ",
    "ДМСБ (малый)": "МСБ",
    "ДМСБ (средний)": "МСБ",
    "ДКБ": "ККБ",
}
SEG_ORDER = ["МкБ", "МСБ", "ККБ"]

# Эталонный набор источников для проверки ИНН как в analysis_crm_segments.ipynb.
ALLOWED_SOURCES_CHECK = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

# Набор источников для самой аналитики (как согласовано: Н2.0 включаем).
ALLOWED_SOURCES_ANALYSIS = ["Н2.0", "Справочник CRM-системы", "CRM-система"]

WINDOWS = {
    "24m": 24,
    "18m": 18,
    "12m": 12,
    "6m": 6,
}


def normalize_inn(s):
    if pd.isna(s):
        return None
    s = str(s).strip()
    if s.endswith(".0"):
        s = s[:-2]
    s = s.lstrip("0") or "0"
    return s.zfill(10) if len(s) <= 10 else s.zfill(12)

In [ ]:
# Загрузка CRM
crm_path = DATA_DIR / "crm_last_version.csv"
df_crm = pd.read_csv(crm_path, dtype=str, low_memory=False)
df_crm.columns = df_crm.columns.str.strip()

df_crm = df_crm.rename(columns={
    "VAL.1": "VAL_1", "DESC_TEXT.1": "DESC_TEXT_1", "ROW_ID.1": "ROW_ID_1",
    "AGREEMENT_NUM.1": "AGREEMENT_NUM_1", "ROW_ID.2": "ROW_ID_2",
    "VAL.2": "VAL_2", "NAME.1": "NAME_1", "VAL.3": "VAL_3", "VAL.5": "VAL_5",
})

# Загрузка defaults
def_path = DATA_DIR / "default_data.pkl"
df_def = pd.read_pickle(def_path)
df_def = df_def.astype(str).replace("nan", np.nan)
df_def.columns = df_def.columns.str.strip()
df_def = df_def.rename(columns={"start": "start_date", "cure": "cure_date", "finish": "finish_date"})

print(f"CRM: {df_crm.shape[0]:,} строк x {df_crm.shape[1]} колонок")
print(f"DEFAULTS: {df_def.shape[0]:,} строк x {df_def.shape[1]} колонок")

## Проверки качества источников: уникальные и неуникальные ИНН

Ниже считаем:
- количество строк;
- количество уникальных ИНН;
- примеры строк с неуникальными ИНН.

Для CRM используем поле `X_INN`, для defaults — `inn`.

In [ ]:
# Подготовка ИНН для проверок (эталонные фильтры)
crm_inn_col = "X_INN"
def_inn_col = "inn"

# --- CRM: фильтры как в analysis_crm_segments.ipynb ---
crm_check = df_crm.copy()
crm_check[crm_inn_col] = crm_check[crm_inn_col].apply(normalize_inn)
crm_check["dt"] = pd.to_datetime(crm_check["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
crm_check = crm_check[(crm_check["dt"] >= DATE_FROM) & (crm_check["dt"] <= DATE_TO)].copy()

if "VAL" in crm_check.columns:
    crm_check["VAL"] = crm_check["VAL"].astype(str)
    crm_check = crm_check[crm_check["VAL"].str.strip().isin(ALLOWED_SOURCES_CHECK)].copy()

crm_check["X_AREA_RESP"] = crm_check["X_AREA_RESP"].astype(str).str.strip()
crm_check["segment"] = crm_check["X_AREA_RESP"].map(SEGMENT_MAP)
crm_check = crm_check[crm_check["segment"].notna()].copy()

crm_check["fp_num"] = crm_check["NUMBER_FP_SFP"].astype(str).str.strip()
crm_check = crm_check.dropna(subset=[crm_inn_col, "NUMBER_FP_SFP"]).copy()
crm_check = crm_check.drop_duplicates(subset=[crm_inn_col, "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()

# --- DEFAULTS: фильтры как в build_dataset.ipynb ---
def_check = df_def.copy()
def_check[def_inn_col] = def_check[def_inn_col].apply(normalize_inn)
def_check["start_date"] = pd.to_datetime(def_check["start_date"], dayfirst=True, errors="coerce")
def_check = (
    def_check
    .dropna(subset=[def_inn_col, "start_date"])
    .query("@DATE_FROM <= start_date <= @DATE_TO")
    .copy()
)

crm_rows = len(crm_check)
crm_unique_inn = crm_check[crm_inn_col].nunique(dropna=True)

def_rows = len(def_check)
def_unique_inn = def_check[def_inn_col].nunique(dropna=True)

print("CRM (после фильтров как в analysis_crm_segments):")
print(f"  строк: {crm_rows:,}")
print(f"  уникальных ИНН (X_INN): {crm_unique_inn:,}")

print("\nDEFAULTS (после фильтров как в build_dataset):")
print(f"  строк: {def_rows:,}")
print(f"  уникальных ИНН (inn): {def_unique_inn:,}")

In [ ]:
# Примеры неуникальных ИНН в CRM (на фильтрованной выборке)
crm_dup_inn = (
    crm_check[crm_inn_col]
    .dropna()
    .value_counts()
    .loc[lambda s: s > 1]
    .head(10)
    .index
)

crm_dup_examples = (
    crm_check[crm_check[crm_inn_col].isin(crm_dup_inn)]
    [[crm_inn_col, "NUMBER_FP_SFP", "IDENTIFICATION_DATE", "VAL", "TYPE_FP", "X_AREA_RESP", "segment"]]
    .sort_values([crm_inn_col, "IDENTIFICATION_DATE"], na_position="last")
    .head(30)
)

print(f"Неуникальных ИНН в CRM после фильтров (кол-во ИНН): {len(crm_dup_inn):,}")
crm_dup_examples

In [ ]:
# Примеры неуникальных ИНН в defaults (на фильтрованной выборке)
def_dup_inn = (
    def_check[def_inn_col]
    .dropna()
    .value_counts()
    .loc[lambda s: s > 1]
    .head(10)
    .index
)

def_dup_examples = (
    def_check[def_check[def_inn_col].isin(def_dup_inn)]
    [[def_inn_col, "start_date", "cure_date", "finish_date", "writeoff", "unlimited_default", "default_reason"]]
    .sort_values([def_inn_col, "start_date"], na_position="last")
    .head(30)
)

print(f"Неуникальных ИНН в DEFAULTS после фильтров (кол-во ИНН): {len(def_dup_inn):,}")
def_dup_examples

## Диагностика потерь ИНН defaults в CRM (воронка фильтров)

Блок показывает, на каком шаге фильтрации CRM теряются ИНН из `default_data.pkl`.

Шаги:
1. Полная CRM (без фильтров)
2. Фильтр по периоду
3. Фильтр по источникам (`Н2.0`, `Справочник CRM-системы`, `CRM-система`)
4. Маппинг сегментов
5. `dropna` по ключевым полям
6. Дедупликация

Для каждого шага выводим:
- сколько ИНН из defaults покрыто в CRM;
- сколько ИНН выбыло относительно предыдущего шага;
- примеры выбывших ИНН.

In [ ]:
# Воронка покрытия ИНН defaults в CRM по шагам фильтрации

default_inn_set = set(def_check[def_inn_col].dropna().unique())

crm_stage0 = df_crm.copy()
crm_stage0["inn"] = crm_stage0["X_INN"].apply(normalize_inn)

crm_stage1 = crm_stage0.copy()
crm_stage1["dt"] = pd.to_datetime(crm_stage1["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
crm_stage1 = crm_stage1[(crm_stage1["dt"] >= DATE_FROM) & (crm_stage1["dt"] <= DATE_TO)].copy()

crm_stage2 = crm_stage1.copy()
crm_stage2["VAL"] = crm_stage2["VAL"].astype(str)
crm_stage2 = crm_stage2[crm_stage2["VAL"].str.strip().isin(ALLOWED_SOURCES_ANALYSIS)].copy()

crm_stage3 = crm_stage2.copy()
crm_stage3["X_AREA_RESP"] = crm_stage3["X_AREA_RESP"].astype(str).str.strip()
crm_stage3["segment"] = crm_stage3["X_AREA_RESP"].map(SEGMENT_MAP)
crm_stage3 = crm_stage3[crm_stage3["segment"].notna()].copy()

crm_stage4 = crm_stage3.copy()
crm_stage4["fp_num"] = crm_stage4["NUMBER_FP_SFP"].astype(str).str.strip()
crm_stage4 = crm_stage4.dropna(subset=["inn", "NUMBER_FP_SFP"]).copy()

crm_stage5 = crm_stage4.drop_duplicates(subset=["inn", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()

stages = [
    ("0_full_crm_no_filters", crm_stage0),
    ("1_after_date_filter", crm_stage1),
    ("2_after_source_filter", crm_stage2),
    ("3_after_segment_mapping", crm_stage3),
    ("4_after_dropna_keys", crm_stage4),
    ("5_after_dedup", crm_stage5),
]

rows = []
prev_cov = None
prev_label = None
for label, d in stages:
    crm_inn_set = set(d["inn"].dropna().unique())
    covered = default_inn_set & crm_inn_set
    dropped_vs_prev = set() if prev_cov is None else (prev_cov - covered)

    rows.append({
        "step": label,
        "crm_rows": len(d),
        "crm_unique_inn": len(crm_inn_set),
        "defaults_total_inn": len(default_inn_set),
        "defaults_in_crm": len(covered),
        "coverage_share": len(covered) / len(default_inn_set) if len(default_inn_set) else np.nan,
        "dropped_vs_prev": len(dropped_vs_prev),
        "dropped_examples": ", ".join(sorted(list(dropped_vs_prev))[:10]) if dropped_vs_prev else "",
        "prev_step": prev_label if prev_label else "-",
    })

    prev_cov = covered
    prev_label = label

coverage_funnel = pd.DataFrame(rows)
print("Воронка покрытия ИНН defaults в CRM:")
display(coverage_funnel)

# Отдельно сохраним выбытия по шагам в длинном виде
funnel_drops_long = []
prev_cov = None
prev_label = None
for label, d in stages:
    crm_inn_set = set(d["inn"].dropna().unique())
    covered = default_inn_set & crm_inn_set
    if prev_cov is not None:
        dropped = sorted(list(prev_cov - covered))
        for inn in dropped:
            funnel_drops_long.append({
                "from_step": prev_label,
                "to_step": label,
                "dropped_inn": inn,
            })
    prev_cov = covered
    prev_label = label

funnel_drops_long = pd.DataFrame(funnel_drops_long)
print("\nПример выбывших ИНН (первые 30):")
display(funnel_drops_long.head(30))

In [ ]:
# Проверка вхождения ИНН: defaults (фильтрованные) -> CRM (фильтрованные)
inn_defaults_set = set(def_check[def_inn_col].dropna().unique())
inn_crm_set = set(crm_check[crm_inn_col].dropna().unique())

inn_in_both = inn_defaults_set & inn_crm_set
inn_missing_in_crm = inn_defaults_set - inn_crm_set

n_def_inn = len(inn_defaults_set)
n_crm_inn = len(inn_crm_set)
n_in_both = len(inn_in_both)
n_missing = len(inn_missing_in_crm)
share_in_both = (n_in_both / n_def_inn) if n_def_inn else np.nan

print("Проверка покрытия ИНН defaults в CRM (после фильтров):")
print(f"  Уникальных ИНН в defaults: {n_def_inn:,}")
print(f"  Уникальных ИНН в CRM:      {n_crm_inn:,}")
print(f"  ИНН defaults, найдено в CRM: {n_in_both:,} ({share_in_both:.2%})")
print(f"  ИНН defaults, отсутствует в CRM: {n_missing:,}")

missing_defaults_vs_crm = pd.DataFrame({"inn": sorted(inn_missing_in_crm)})
print("\nПервые 30 ИНН из defaults, отсутствующих в CRM:")
display(missing_defaults_vs_crm.head(30))

In [ ]:
# Подготовка defaults (default_date на уровне ИНН) как в build_dataset.ipynb
severity_priority = {"тяжёлый": 0, "активный": 1, "надзор": 2, "вышел из дефолта": 3}

def classify_severity(row):
    if row.get("unlimited_default") == 1:
        return "тяжёлый"
    if pd.notna(row.get("writeoff")) or pd.notna(row.get("liquidation")):
        return "тяжёлый"
    if pd.notna(row.get("finish_date")):
        return "вышел из дефолта"
    if pd.notna(row.get("cure_date")):
        return "надзор"
    return "активный"

df_def_work = df_def.copy()
df_def_work["inn"] = df_def_work["inn"].apply(normalize_inn)
for col in ["start_date", "cure_date", "finish_date", "writeoff", "liquidation"]:
    if col in df_def_work.columns:
        df_def_work[col] = pd.to_datetime(df_def_work[col], dayfirst=True, errors="coerce")

if "unlimited_default" in df_def_work.columns:
    df_def_work["unlimited_default"] = pd.to_numeric(df_def_work["unlimited_default"], errors="coerce").fillna(0).astype(int)
else:
    df_def_work["unlimited_default"] = 0

df_def_work["severity"] = df_def_work.apply(classify_severity, axis=1)
df_def_work["_sev_rank"] = df_def_work["severity"].map(severity_priority)

defaults = (
    df_def_work
    .dropna(subset=["inn", "start_date"])
    .query("@DATE_FROM <= start_date <= @DATE_TO")
    .sort_values("_sev_rank")
    .groupby("inn", as_index=False)
    .agg(
        default_date=("start_date", "min"),
        severity=("severity", "first"),
        writeoff=("writeoff", "min"),
        liquidation=("liquidation", "min"),
        finish_date=("finish_date", "min"),
        cure_date=("cure_date", "min"),
        unlimited_default=("unlimited_default", "max"),
    )
)
defaults["default_flag"] = 1

print(f"Дефолтных ИНН в когорте: {len(defaults):,}")
print(f"Период default_date: {defaults['default_date'].min()} — {defaults['default_date'].max()}")

In [ ]:
# Подготовка CRM ФП для аналитики (с Н2.0) + сегментация через X_AREA_RESP и fallback на VAL.4
df_crm_work = df_crm.copy()
df_crm_work["IDENTIFICATION_DATE"] = pd.to_datetime(df_crm_work["IDENTIFICATION_DATE"], dayfirst=True, errors="coerce")
df_crm_work["X_INN"] = df_crm_work["X_INN"].apply(normalize_inn)

df_crm_work = df_crm_work[
    (df_crm_work["IDENTIFICATION_DATE"] >= DATE_FROM) &
    (df_crm_work["IDENTIFICATION_DATE"] <= DATE_TO)
].copy()

df_crm_work["VAL"] = df_crm_work["VAL"].astype(str).str.strip()
df_crm_work = df_crm_work[df_crm_work["VAL"].isin(ALLOWED_SOURCES_ANALYSIS)].copy()

df_crm_work["TYPE_FP"] = df_crm_work["TYPE_FP"].astype(str).str.strip().replace({"FP": "ФП", "SFP": "СФП"})
df_crm_work["X_AREA_RESP"] = df_crm_work["X_AREA_RESP"].astype(str).str.strip()
df_crm_work["segment_zo"] = df_crm_work["X_AREA_RESP"].map(SEGMENT_MAP)

val4_col = "VAL.4" if "VAL.4" in df_crm_work.columns else ("VAL_4" if "VAL_4" in df_crm_work.columns else None)
if val4_col is None:
    raise KeyError("Не найдена колонка 'VAL.4'/'VAL_4' в CRM")

val4_map = {
    "Микро": "МкБ",
    "Малый": "МСБ",
    "Средний": "МСБ",
    "Крупный": "ККБ",
    "Крупнейший": "ККБ",
    "Не подл. сегм.": np.nan,
}
df_crm_work["val4_raw"] = df_crm_work[val4_col].astype(str).str.strip()
df_crm_work["segment_val4"] = df_crm_work["val4_raw"].map(val4_map)

df_crm_work["fp_num"] = df_crm_work["NUMBER_FP_SFP"].astype(str).str.strip()
df_crm_work = df_crm_work.dropna(subset=["X_INN", "NUMBER_FP_SFP"]).copy()
df_crm_work = df_crm_work.drop_duplicates(subset=["X_INN", "fp_num", "TYPE_FP", "IDENTIFICATION_DATE"]).copy()

fp_raw = (
    df_crm_work[["X_INN", "NUMBER_FP_SFP", "TYPE_FP", "IDENTIFICATION_DATE", "segment_zo", "segment_val4"]]
    .rename(columns={
        "X_INN": "inn",
        "NUMBER_FP_SFP": "fp_sfp_code",
        "TYPE_FP": "type_fp",
        "IDENTIFICATION_DATE": "fp_date",
    })
    .dropna(subset=["inn", "fp_sfp_code", "fp_date"])
)
fp_raw["fp_sfp_code"] = fp_raw["fp_sfp_code"].astype(str).str.strip()
fp_raw["type_fp"] = fp_raw["type_fp"].astype(str).str.strip().replace({"": np.nan})

print(f"FP-записей после фильтров (до назначения финального сегмента): {len(fp_raw):,}")

In [ ]:
# Сегмент компании: приоритет X_AREA_RESP; fallback на VAL.4,
# если сегмент по X_AREA_RESP отсутствует ИЛИ не входит в МкБ/МСБ/ККБ.

def _mode(series):
    m = series.mode(dropna=True)
    return m.iloc[0] if len(m) else np.nan

seg_zo = (
    fp_raw[["inn", "segment_zo"]]
    .dropna(subset=["segment_zo"])
    .groupby("inn", as_index=False)
    .agg(segment_zo=("segment_zo", _mode))
)

seg_val4 = (
    fp_raw[["inn", "segment_val4"]]
    .dropna(subset=["segment_val4"])
    .groupby("inn", as_index=False)
    .agg(segment_val4=("segment_val4", _mode))
)

seg_company = (
    defaults[["inn"]]
    .drop_duplicates()
    .merge(seg_zo, on="inn", how="left")
    .merge(seg_val4, on="inn", how="left")
)

valid_zo = seg_company["segment_zo"].isin(SEG_ORDER)
seg_company["segment"] = np.where(valid_zo, seg_company["segment_zo"], seg_company["segment_val4"])

# Формируем рабочую таблицу ФП с финальным сегментом компании
fp = (
    fp_raw
    .merge(seg_company[["inn", "segment"]], on="inn", how="left")
    .dropna(subset=["segment"])
    [["inn", "fp_sfp_code", "type_fp", "fp_date", "segment"]]
    .copy()
)

# Когорта дефолтных ИНН + сегмент (final)
cohort = defaults.merge(seg_company[["inn", "segment"]], on="inn", how="left")
cohort = cohort[cohort["segment"].isin(SEG_ORDER)].copy()

n_zo = seg_company["segment_zo"].isin(SEG_ORDER).sum()
n_val4 = seg_company["segment_val4"].isin(SEG_ORDER).sum()
n_final = seg_company["segment"].isin(SEG_ORDER).sum()
n_fallback_only = ((~seg_company["segment_zo"].isin(SEG_ORDER)) & (seg_company["segment_val4"].isin(SEG_ORDER))).sum()

print("Покрытие сегментации defaults-ИНН:")
print(f"  по X_AREA_RESP (валидный маппинг в МкБ/МСБ/ККБ): {n_zo:,}")
print(f"  по VAL.4 (валидный маппинг в МкБ/МСБ/ККБ):       {n_val4:,}")
print(f"  final (ZO -> VAL.4): {n_final:,}")
print(f"  из них через fallback VAL.4 (ZO невалиден/пуст): {n_fallback_only:,}")

print(f"\nFP-записей после присвоения финального сегмента: {len(fp):,}")
print("Сегменты в FP:")
print(fp["segment"].value_counts().reindex(SEG_ORDER, fill_value=0).to_string())

print(f"\nКогорта после присвоения сегментов: {len(cohort):,}")
print(cohort["segment"].value_counts().reindex(SEG_ORDER, fill_value=0).to_string())

In [ ]:
# База событий ФП относительно default_date
base = fp.merge(cohort[["inn", "default_date", "segment"]], on=["inn", "segment"], how="inner")

# Разница в днях: сколько дней до дефолта (включая день дефолта = 0)
base["days_before_default"] = (base["default_date"] - base["fp_date"]).dt.days
base = base[base["days_before_default"] >= 0].copy()

print(f"Событий ФП в дефолтной когорте (до/в день дефолта): {len(base):,}")

In [ ]:
# Формирование пулов ФП/СФП на уровне ИНН x окно (с None)
rows_pool = []
rows_long = []

for window_name, months in WINDOWS.items():
    left = base["default_date"] - pd.DateOffset(months=months)
    mask = (base["fp_date"] >= left) & (base["fp_date"] <= base["default_date"])
    w = base[mask].copy()

    # уникальные факторы в окне на ИНН
    uniq = (
        w[["inn", "segment", "fp_sfp_code", "type_fp"]]
        .drop_duplicates()
    )

    # список факторов по ИНН
    pool = (
        uniq.groupby(["inn", "segment"], as_index=False)
        .agg(fp_pool=("fp_sfp_code", lambda s: sorted(set(map(str, s)))))
    )

    # гарантируем, что каждый ИНН когорты присутствует (None, если пусто)
    pool_full = cohort[["inn", "segment"]].merge(pool, on=["inn", "segment"], how="left")
    pool_full["window"] = window_name
    pool_full["fp_pool"] = pool_full["fp_pool"].apply(lambda x: x if isinstance(x, list) else None)
    rows_pool.append(pool_full)

    uniq["window"] = window_name
    rows_long.append(uniq)

pool_by_inn_window = pd.concat(rows_pool, ignore_index=True)
long_for_top = pd.concat(rows_long, ignore_index=True)

print("Пул по ИНН x окно:")
print(pool_by_inn_window.head(10))
print(f"\nВсего строк в пуле: {len(pool_by_inn_window):,}")

In [ ]:
# TOP-10 по сегментам и окнам
# count_inn: число уникальных ИНН с фактором
# share_in_segment_defaults: доля от ВСЕХ дефолтных ИНН сегмента в окне (включая None)

denom = (
    pool_by_inn_window
    .groupby(["window", "segment"], as_index=False)
    .agg(total_defaults=("inn", "nunique"))
)

top_base = (
    long_for_top
    .groupby(["window", "segment", "fp_sfp_code", "type_fp"], as_index=False)
    .agg(count_inn=("inn", "nunique"))
    .merge(denom, on=["window", "segment"], how="left")
)
top_base["share_in_segment_defaults"] = top_base["count_inn"] / top_base["total_defaults"]

top_base = top_base.sort_values(
    ["window", "segment", "count_inn", "share_in_segment_defaults", "fp_sfp_code"],
    ascending=[True, True, False, False, True]
)

top_base["rank"] = top_base.groupby(["window", "segment"]).cumcount() + 1
top10_windows_all = top_base[top_base["rank"] <= 10].copy()

print("TOP-10 (превью):")
display(top10_windows_all.head(30))

In [ ]:
# Проверки качества расчета

# 1) Полнота пула: на каждый window должно быть столько строк, сколько ИНН в когорте
expected_n = cohort["inn"].nunique()
check_pool = (
    pool_by_inn_window
    .groupby("window", as_index=False)
    .agg(n_inn=("inn", "nunique"), n_rows=("inn", "size"))
)
check_pool["expected_n"] = expected_n
check_pool["ok"] = (check_pool["n_inn"] == expected_n) & (check_pool["n_rows"] == expected_n)

# 2) Покрытие None по окнам
none_stats = (
    pool_by_inn_window
    .assign(is_none=lambda x: x["fp_pool"].isna())
    .groupby(["window", "segment"], as_index=False)
    .agg(total_defaults=("inn", "nunique"), none_count=("is_none", "sum"))
)
none_stats["none_share"] = none_stats["none_count"] / none_stats["total_defaults"]

# 3) Уникальность рангов
rank_dups = top10_windows_all.duplicated(subset=["window", "segment", "rank"]).sum()

print("Проверка полноты пула (ИНН по окнам):")
display(check_pool)

print("\nСтатистика None по окнам и сегментам:")
display(none_stats.sort_values(["window", "segment"]))

print(f"\nДубликатов по ключу (window, segment, rank): {rank_dups}")

In [ ]:
# Экспорт результатов
out_xlsx = DATA_DIR / "Приложение к отчету TOP10_FPSFP_по_окнам_до_default_date.xlsx"
out_missing_xlsx = DATA_DIR / "ИНН_defaults_которых_нет_в_CRM.xlsx"

# Подбираем доступный движок для ExcelWriter.
# В первую очередь openpyxl (обычно есть в окружениях Jupyter), затем xlsxwriter.
_engine = None
try:
    import openpyxl  # noqa: F401
    _engine = "openpyxl"
except Exception:
    try:
        import xlsxwriter  # noqa: F401
        _engine = "xlsxwriter"
    except Exception:
        _engine = None

_writer_kwargs = {"engine": _engine} if _engine else {}

# 1) Основной файл с результатами аналитики
with pd.ExcelWriter(out_xlsx, **_writer_kwargs) as writer:
    # Сводные листы
    top10_windows_all.to_excel(writer, sheet_name="top10_windows_all", index=False)

    pool_export = pool_by_inn_window.copy()
    pool_export["fp_pool"] = pool_export["fp_pool"].apply(lambda x: None if x is None else ", ".join(x))
    pool_export.to_excel(writer, sheet_name="pool_by_inn_window", index=False)

    check_pool.to_excel(writer, sheet_name="check_pool_fullness", index=False)
    none_stats.to_excel(writer, sheet_name="none_stats", index=False)

    # Диагностика потерь ИНН defaults в CRM
    coverage_funnel.to_excel(writer, sheet_name="coverage_funnel", index=False)
    funnel_drops_long.to_excel(writer, sheet_name="funnel_drops_long", index=False)

    # Таблица финальной сегментации defaults-ИНН (ZO с fallback на VAL.4)
    if "seg_company" in globals():
        seg_company.to_excel(writer, sheet_name="segment_assignment", index=False)

    # Сравнение сегментации X_AREA_RESP vs VAL.4 (если блок уже был посчитан)
    if "segment_compare_by_inn" in globals():
        segment_compare_by_inn.to_excel(writer, sheet_name="segment_compare_inn", index=False)
    if "segment_compare_cross" in globals():
        segment_compare_cross.to_excel(writer, sheet_name="segment_compare_cross")

    # Листы по каждому окну и сегменту
    for w in WINDOWS.keys():
        for seg in SEG_ORDER:
            sub = top10_windows_all[(top10_windows_all["window"] == w) & (top10_windows_all["segment"] == seg)].copy()
            sheet = f"top10_{w}_{seg}"[:31]
            sub.to_excel(writer, sheet_name=sheet, index=False)

# 2) Отдельный файл: ИНН из defaults, которых нет в CRM
#    a) относительно CRM после фильтров (текущая рабочая логика)
missing_filtered = missing_defaults_vs_crm.copy()
missing_filtered["scope"] = "filtered_crm"

#    b) относительно полной CRM без фильтров
full_crm_inn_set = set(crm_stage0["inn"].dropna().unique())
missing_full = pd.DataFrame({"inn": sorted(default_inn_set - full_crm_inn_set)})
missing_full["scope"] = "full_crm_no_filters"

with pd.ExcelWriter(out_missing_xlsx, **_writer_kwargs) as writer:
    missing_filtered.to_excel(writer, sheet_name="missing_vs_filtered", index=False)
    missing_full.to_excel(writer, sheet_name="missing_vs_full", index=False)

# Дополнительно отдельный файл строго для "нет в CRM до фильтров"
out_missing_only_xlsx = DATA_DIR / "ИНН_defaults_отсутствуют_в_CRM_до_фильтров.xlsx"
missing_full.to_excel(out_missing_only_xlsx, index=False)

print(f"Экспорт сохранен: {out_xlsx} (engine={_engine or 'auto'})")
print(f"Отдельный файл с отсутствующими ИНН: {out_missing_xlsx}")
print(f"Отдельный файл (только нет в CRM до фильтров): {out_missing_only_xlsx}")

## Проверка расхождений сегментации: `X_AREA_RESP` vs `VAL.4`

Сравниваем сегмент на уровне ИНН двумя способами:
- по ЗО (`X_AREA_RESP` -> `segment`),
- по итоговому сегменту `VAL.4` с маппингом:
  - `Микро` -> `МкБ`
  - `Малый`/`Средний` -> `МСБ`
  - `Крупный`/`Крупнейший` -> `ККБ`
  - `Не подл. сегм.` -> `None`

Сравнение делаем для той же CRM-выборки, где получено 9 919 ИНН (`crm_check`).

In [ ]:
# Диагностика расхождений X_AREA_RESP vs VAL.4 на уровне ИНН
val4_col = "VAL.4" if "VAL.4" in crm_check.columns else ("VAL_4" if "VAL_4" in crm_check.columns else None)
if val4_col is None:
    raise KeyError("Не найдена колонка 'VAL.4'/'VAL_4' в crm_check")

val4_map = {
    "Микро": "МкБ",
    "Малый": "МСБ",
    "Средний": "МСБ",
    "Крупный": "ККБ",
    "Крупнейший": "ККБ",
    "Не подл. сегм.": np.nan,
}

def _mode_safe(s):
    m = s.dropna().mode()
    return m.iloc[0] if len(m) else np.nan

cmp_df = crm_check[["X_INN", "segment", val4_col]].copy()
cmp_df["val4_raw"] = cmp_df[val4_col].astype(str).str.strip()
cmp_df["segment_val4"] = cmp_df["val4_raw"].map(val4_map)

# Агрегация на уровень ИНН
cmp_inn = (
    cmp_df.groupby("X_INN", as_index=False)
    .agg(
        segment_zo=("segment", _mode_safe),
        segment_val4=("segment_val4", _mode_safe),
        zo_variants=("segment", lambda x: x.dropna().nunique()),
        val4_variants=("segment_val4", lambda x: x.dropna().nunique()),
        val4_raw_variants=("val4_raw", lambda x: ", ".join(sorted(set(x.dropna()))[:8])),
    )
)

cmp_inn["compare_status"] = np.where(
    cmp_inn["segment_val4"].isna(),
    "missing_val4_segment",
    np.where(cmp_inn["segment_zo"] == cmp_inn["segment_val4"], "match", "mismatch")
)

print("Сравнение сегментов на уровне ИНН (X_AREA_RESP vs VAL.4):")
print(f"  ИНН в сравнении: {len(cmp_inn):,}")
print("  Статусы:")
print(cmp_inn["compare_status"].value_counts(dropna=False).to_string())

print("\nИНН с неоднозначным сегментом внутри ИНН:")
print(f"  по X_AREA_RESP (zo_variants > 1): {(cmp_inn['zo_variants'] > 1).sum():,}")
print(f"  по VAL.4 (val4_variants > 1): {(cmp_inn['val4_variants'] > 1).sum():,}")

mismatch_inn = cmp_inn[cmp_inn["compare_status"] == "mismatch"].copy()
missing_val4_inn = cmp_inn[cmp_inn["compare_status"] == "missing_val4_segment"].copy()

print("\nПримеры mismatch (первые 30):")
display(mismatch_inn.head(30))

print("\nПримеры missing_val4_segment (первые 30):")
display(missing_val4_inn.head(30))

# Кросс-таблица для наглядности
cross = pd.crosstab(cmp_inn["segment_zo"], cmp_inn["segment_val4"], dropna=False)
print("\nКросс-таблица segment_zo x segment_val4:")
display(cross)

# Доступно для экспорта при необходимости
segment_compare_by_inn = cmp_inn.copy()
segment_compare_cross = cross.copy()

## ИНН из defaults, отсутствующие в полной CRM (до фильтрации)

Этот блок формирует список ИНН из `default_data.pkl`, которых нет в `crm_last_version.csv` даже в полной выгрузке (до любых фильтров).

Ожидаемое количество по вашей проверке: `246 - 211 = 35` ИНН.

In [ ]:
# Список ИНН из defaults, отсутствующих в полной CRM (до фильтров)
full_crm_inn_set = set(crm_stage0["inn"].dropna().unique())
missing_defaults_full_crm = pd.DataFrame({
    "inn": sorted(default_inn_set - full_crm_inn_set)
})

print("ИНН из defaults, отсутствующие в полной CRM (до фильтров):")
print(f"  defaults ИНН всего: {len(default_inn_set):,}")
print(f"  покрыто в полной CRM: {len(default_inn_set & full_crm_inn_set):,}")
print(f"  отсутствует в полной CRM: {len(missing_defaults_full_crm):,}")

display(missing_defaults_full_crm)

# Отдельный Excel-файл только с этим списком
out_missing_only_xlsx = DATA_DIR / "ИНН_defaults_отсутствуют_в_CRM_до_фильтров.xlsx"
missing_defaults_full_crm.to_excel(out_missing_only_xlsx, index=False)
print(f"\nСохранено в файл: {out_missing_only_xlsx}")